## Preprocess OSM into parquets

## Embeddings for semantic search

In [ ]:
import pandas as pd
import geopandas as gpd

df = gpd.read_parquet(rf"E:\Data\query-earth\embeddings\osm\2026\natural.parquet")

In [ ]:
df

,lat,lon,natural,geometry
0,37.144093,-121.984710,saddle,POINT (-121.98471 37.14409)
1,34.204039,-117.808757,saddle,POINT (-117.80876 34.20404)
2,41.738128,-121.529074,volcano,POINT (-121.52907 41.73813)
3,41.728373,-121.548889,cave_entrance,POINT (-121.54889 41.72837)
4,37.865539,-122.243419,stone,POINT (-122.24342 37.86554)
...,...,...,...,...
852648,NaN,NaN,scree,"MULTIPOLYGON (((-119.87783 39.34051, -119.8778..."
852649,NaN,NaN,scrub,"MULTIPOLYGON (((-119.94491 39.05863, -119.9449..."
852650,NaN,NaN,bare_rock,"MULTIPOLYGON (((-119.94509 39.05859, -119.9450..."
852651,NaN,NaN,scrub,"MULTIPOLYGON (((-119.94477 39.0774, -119.94485..."


In [ ]:
unique_classes = [str(i) for i in df.dropna(subset=['natural']).natural.unique()]

In [ ]:
print(len(unique_classes), unique_classes)

117 ['saddle', 'volcano', 'cave_entrance', 'stone', 'peak', 'tree', 'spring', 'cliff', 'rock', 'hot_spring', 'cape', 'wood', 'bay', 'arch', 'rock_formation', 'beach', 'water', 'crater', 'geyser', 'sinkhole', 'hill', 'slope', 'heath', 'wetland', 'desert', 'plateau', 'tree_stump', 'grassland', 'valley', 'point', 'scrub', 'scree', 'canyon', 'ridge', 'dune', 'yes', 'bush', 'ravine', 'grove', 'mountain_range', 'shrub', 'flat', 'wildflowers', 'succulent_plant', 'cactus', 'flowering_plant', 'bare_rock', 'geothermal_area', 'cirque', 'plant', 'landform', 'birds_nest', 'peninsula', 'mesa', 'grass', 'butte', 'basin', 'stump', 'sediment', 'hills', 'plain', 'depression', 'cave', 'caldera', 'fumarole', 'agave', 'coastline', 'shingle', 'reef', 'sand', 'mud', 'fell', 'shrubbery', 'wadi', 'dry wash', 'strait', 'glacier', 'knoll', 'gully', 'gorge', 'dry_wash', 'shoal', 'tree_row', 'mountain_basin', 'lava', 'fault', 'range', 'landslide', 'shrubland', 'arete', 'meadow', 'tree_group', 'land', 'boulder', 'd

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer

# 2. Load the all-MiniLM-L6-v2 model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Generate embeddings (convert to Python lists for clean Parquet serialization)
embeddings = model.encode(unique_classes, convert_to_numpy=True).tolist()

# 4. Construct the pandas DataFrame matching your target structure
df = pd.DataFrame({"amenities": unique_classes, "embedding": embeddings})

# 5. Save to a Parquet file
parquet_filename = rf"E:\Data\query-earth\embeddings\osm\2026\poi_semnatic.parquet"
df.to_parquet(parquet_filename, index=False)

print(f"Successfully saved {len(df)} embeddings to '{parquet_filename}'.")
print("\nFirst 2 rows preview:")
print(df.head(2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully saved 584 embeddings to 'E:\Data\query-earth\embeddings\osm\2026\poi_semnatic.parquet'.

First 2 rows preview:
  amenities                                          embedding
0      bank  [0.00487261638045311, 0.025024818256497383, -0...
1   toilets  [0.0520838238298893, 0.03292398899793625, 0.00...


In [ ]:
pd.read_parquet(rf"D:\Code\query-earth\src\embeddings\poi_embeddings.parquet")

,amenities,embedding
0,stream,"[-0.049106162, -0.041543964, -0.053305697, -0...."
1,borderland,"[0.08206559, 0.03463122, -0.08410268, -0.00889..."
2,knoll,"[-0.014341307, -0.033093102, 0.040650368, -0.0..."
3,canyon,"[0.0007149489, 0.018037673, -0.036527846, 0.05..."
4,park,"[0.04683362, 0.033664923, 0.028864298, -0.0315..."
...,...,...
714,massage,"[-0.06918556, 0.015118147, 0.037174415, 0.0665..."
715,feeding_place,"[0.006678588, 0.00031741234, -0.055687092, 0.0..."
716,mikveh,"[-0.11270124, 0.083171666, 0.008068371, 0.0276..."
717,makerspace,"[-0.05816774, -0.078223355, 0.011646609, 0.029..."


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the Parquet file and the model
df = pd.read_parquet(parquet_filename)
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Define your search query and top-K count
search_query = "emergency serives"
top_k = 15

# 3. Generate embedding for the search query
query_embedding = model.encode([search_query], convert_to_numpy=True)

# 4. Calculate cosine similarity against stored embeddings
stored_embeddings = df["embedding"].tolist()
similarities = cosine_similarity(query_embedding, stored_embeddings)[0]

# 5. Add similarity scores to DataFrame and retrieve top K results
df["similarity"] = similarities
top_results = df.nlargest(top_k, "similarity")[["amenities", "similarity"]]

# Display results
print(f"Top {top_k} results for query: '{search_query}'\n")
print(top_results.to_string(index=False))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top 15 results for query: 'emergency serives'

                               amenities  similarity
                             urgent_care    0.484460
                           air_ambulance    0.479867
                                hospital    0.430942
                        first_aid_school    0.412686
                               first_aid    0.398327
                             crematorium    0.388058
                     fire_station;prison    0.376305
                                 service    0.370967
                                  prison    0.370141
                      crematory_services    0.367695
                                 toilets    0.359046
                                    stor    0.354631
                          disused:clinic    0.354610
                                     spa    0.349121
assisted_living;skilled_nursing_facility    0.345494


### Appdx

In [ ]:
import os
import gc
from pyrosm import OSM

# Paths
pbf_path = r"E:\Data\query-earth\osm\california-latest.osm.pbf"
output_dir = r"E:\Data\query-earth\osm\2026"

# Create output folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Initialize OSM parser
osm = OSM(pbf_path)

# --- 1. BUILDINGS ---
print("Extracting buildings...")
buildings = osm.get_buildings()
if buildings is not None and not buildings.empty:
    out_path = os.path.join(output_dir, "buildings.parquet")
    print(f"Saving to {out_path}...")
    buildings.to_parquet(out_path, compression="snappy")
    del buildings
    gc.collect()

# --- 2. LANDUSE ---
print("Extracting landuse...")
landuse = osm.get_landuse()
if landuse is not None and not landuse.empty:
    out_path = os.path.join(output_dir, "landuse.parquet")
    print(f"Saving to {out_path}...")
    landuse.to_parquet(out_path, compression="snappy")
    del landuse
    gc.collect()

# --- 3. NATURAL FEATURES ---
print("Extracting natural features...")
natural = osm.get_natural()
if natural is not None and not natural.empty:
    out_path = os.path.join(output_dir, "natural.parquet")
    print(f"Saving to {out_path}...")
    natural.to_parquet(out_path, compression="snappy")
    del natural
    gc.collect()

# --- 4. ROADS & NETWORKS ---
print("Extracting roads...")
roads = osm.get_network(network_type="all")
if roads is not None and not roads.empty:
    out_path = os.path.join(output_dir, "roads.parquet")
    print(f"Saving to {out_path}...")
    roads.to_parquet(out_path, compression="snappy")
    del roads
    gc.collect()

# --- 5. WATERWAYS ---
print("Extracting waterways...")
waterways = osm.get_data_by_custom_criteria(custom_filter={"waterway": True})
if waterways is not None and not waterways.empty:
    out_path = os.path.join(output_dir, "waterways.parquet")
    print(f"Saving to {out_path}...")
    waterways.to_parquet(out_path, compression="snapingpy")
    del waterways
    gc.collect()

Extracting buildings...
Saving to E:\Data\query-earth\osm\2026\buildings.parquet...
Extracting landuse...
Saving to E:\Data\query-earth\osm\2026\landuse.parquet...
Extracting natural features...
Saving to E:\Data\query-earth\osm\2026\natural.parquet...
Extracting roads...
Saving to E:\Data\query-earth\osm\2026\roads.parquet...
Extracting waterways...
Saving to E:\Data\query-earth\osm\2026\waterways.parquet...


In [ ]:
def prepare_gdf_for_parquet(gdf):
    """Converts mixed object columns (like 'id') to string to avoid PyArrow serialization errors."""
    if gdf is None or gdf.empty:
        return gdf
    
    # Cast the 'id' column explicitly to string if present
    if "id" in gdf.columns:
        gdf["id"] = gdf["id"].astype(str)
        
    # Convert any other object/mixed columns (excluding geometry) to string type
    for col in gdf.select_dtypes(include=["object"]).columns:
        if col != gdf._geometry_column_name:
            gdf[col] = gdf[col].astype(str)
            
    return gdf
    
# 6. POINTS OF INTEREST (POIs)
print("Extracting POIs...")
pois = prepare_gdf_for_parquet(
    osm.get_pois(
        custom_filter={
            "amenity": True,
            "shop": True,
            "leisure": True,
            "tourism": True,
            "place": True
        }
    )
)
if pois is not None:
    out_path = os.path.join(output_dir, "pois.parquet")
    print(f"Saving to {out_path}...")
    pois.to_parquet(out_path, compression="snappy")
    del pois
    gc.collect()

print("\nAll layers successfully extracted and saved as GeoParquet files.")

Extracting POIs...


C:\Users\sus14836\AppData\Local\Temp\ipykernel_41376\3674067905.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in gdf.select_dtypes(include=["object"]).columns:


Saving to E:\Data\query-earth\osm\2026\pois.parquet...

All layers successfully extracted and saved as GeoParquet files.


### Landuse

In [ ]:
import pandas as pd
import geopandas as gpd

df = gpd.read_parquet(rf"E:\Data\query-earth\osm\2026\landuse.parquet")

In [ ]:
df

,lat,timestamp,version,visible,lon,changeset,id,tags,construction,depot,...,commercial,farmland,farmyard,grass,meadow,quarry,railway,recreation_ground,retail,orchard
0,35.325313,1510540681,5,False,-120.727476,0.0,91597312,"{""access"":""private""}",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,35.325196,1510540681,4,False,-120.727364,0.0,91597369,"{""access"":""private""}",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,38.617960,1692739405,4,False,-121.400786,0.0,150954852,"{""ele"":""20"",""gnis:feature_id"":""1660019"",""name""...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,34.071124,1756101978,4,False,-117.181427,0.0,150980949,"{""ele"":""419"",""gnis:feature_id"":""1839597"",""name...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,40.776775,1738211854,2,False,-122.283016,0.0,320120937,"{""name"":""Shasta Iron Mine"",""wikidata"":""Q494479...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
518351,NaN,1784660365,1,None,NaN,0.0,21127188,"{""type"":""multipolygon"",""members"":[{""member_id""...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
518352,NaN,1784662801,1,None,NaN,0.0,21127290,"{""type"":""multipolygon"",""members"":[{""member_id""...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
518353,NaN,1784682964,1,None,NaN,0.0,21128146,"{""ownership"":""county"",""type"":""multipolygon"",""m...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
518354,NaN,1784941843,1,None,NaN,0.0,21141560,"{""type"":""multipolygon"",""members"":[{""member_id""...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df = df[['lat', 'lon', 'landuse', 'geometry']]

In [ ]:
df[df.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].to_parquet(rf"E:\Data\query-earth\embeddings\osm\2026\landuse.parquet")

### Buildings

In [ ]:
import pandas as pd
import geopandas as gpd

df = gpd.read_parquet(rf"E:\Data\query-earth\osm\2026\buildings.parquet")

In [ ]:
df = df[['amenity', 'name', 'geometry']]

In [ ]:
df

,amenity,name,geometry
0,NaN,Hangar 1,"POLYGON ((-122.05507 37.4141, -122.05508 37.41..."
1,NaN,NaN,"POLYGON ((-117.5651 33.42527, -117.5651 33.425..."
2,NaN,NaN,"POLYGON ((-117.45199 34.42306, -117.45191 34.4..."
3,NaN,NaN,"POLYGON ((-117.52246 34.41063, -117.52254 34.4..."
4,NaN,NaN,"POLYGON ((-117.50382 34.40248, -117.50394 34.4..."
...,...,...,...
10277113,NaN,NaN,"MULTIPOLYGON (((-120.98514 38.70054, -120.9851..."
10277114,NaN,Augie's,"POLYGON ((-119.6978 34.41905, -119.69786 34.41..."
10277115,NaN,NaN,"POLYGON ((-122.49725 37.74987, -122.49723 37.7..."
10277116,NaN,NaN,"POLYGON ((-117.21069 33.93341, -117.21069 33.9..."


In [ ]:
df.to_parquet(rf"E:\Data\query-earth\embeddings\osm\2026\buildings.parquet")

### Natural

In [ ]:
import pandas as pd
import geopandas as gpd

df = gpd.read_parquet(rf"E:\Data\query-earth\osm\2026\natural.parquet")

In [ ]:
df

,lat,timestamp,version,visible,lon,changeset,id,tags,natural,peak,...,beach,grassland,reef,sand,scrub,stone,tree_row,wood,rock,valley
0,37.144093,1676333776,50,False,-121.984710,0.0,26497399,"{""ele"":""548"",""mountain_pass"":""yes"",""name"":""Pat...",saddle,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,34.204039,1609808614,7,False,-117.808757,0.0,26794832,"{""name"":""Horse Canyon Saddle""}",saddle,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,41.738128,1506287239,5,False,-121.529074,0.0,33113071,"{""ele"":""1618"",""name"":""Schonchin Butte"",""volcan...",volcano,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,41.728373,1738129819,3,False,-121.548889,0.0,33113431,"{""created_by"":""JOSM"",""name"":""Merrill Ice Cave""...",cave_entrance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,37.865539,1691917308,5,False,-122.243419,0.0,34351134,"{""name"":""The Rock"",""tourism"":""viewpoint""}",stone,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
852648,NaN,1784682964,1,None,NaN,0.0,21128147,"{""ownership"":""national"",""type"":""multipolygon"",...",scree,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
852649,NaN,1784920941,1,None,NaN,0.0,21140696,"{""ownership"":""national"",""type"":""multipolygon"",...",scrub,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
852650,NaN,1784920941,1,None,NaN,0.0,21140697,"{""ownership"":""national"",""type"":""multipolygon"",...",bare_rock,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
852651,NaN,1784953733,1,None,NaN,0.0,21142069,"{""ownership"":""private"",""type"":""multipolygon"",""...",scrub,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 1. Ensure natural column is string, drop NaNs if needed
has_no_digits = (
    df['natural']
    .fillna('')
    .astype(str)
    .str.contains(r'\d', regex=True) == False
)

# 2. Filter out empty strings/NaNs (optional, depending on whether you want to keep missing natural values)
has_value = df['natural'].notna() & (df['natural'] != '')

# 3. Apply filter
df = df[has_no_digits & has_value].copy()

In [ ]:
df

,lat,timestamp,version,visible,lon,changeset,id,tags,natural,peak,...,beach,grassland,reef,sand,scrub,stone,tree_row,wood,rock,valley
0,37.144093,1676333776,50,False,-121.984710,0.0,26497399,"{""ele"":""548"",""mountain_pass"":""yes"",""name"":""Pat...",saddle,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,34.204039,1609808614,7,False,-117.808757,0.0,26794832,"{""name"":""Horse Canyon Saddle""}",saddle,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,41.738128,1506287239,5,False,-121.529074,0.0,33113071,"{""ele"":""1618"",""name"":""Schonchin Butte"",""volcan...",volcano,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,41.728373,1738129819,3,False,-121.548889,0.0,33113431,"{""created_by"":""JOSM"",""name"":""Merrill Ice Cave""...",cave_entrance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,37.865539,1691917308,5,False,-122.243419,0.0,34351134,"{""name"":""The Rock"",""tourism"":""viewpoint""}",stone,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
852648,NaN,1784682964,1,None,NaN,0.0,21128147,"{""ownership"":""national"",""type"":""multipolygon"",...",scree,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
852649,NaN,1784920941,1,None,NaN,0.0,21140696,"{""ownership"":""national"",""type"":""multipolygon"",...",scrub,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
852650,NaN,1784920941,1,None,NaN,0.0,21140697,"{""ownership"":""national"",""type"":""multipolygon"",...",bare_rock,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
852651,NaN,1784953733,1,None,NaN,0.0,21142069,"{""ownership"":""private"",""type"":""multipolygon"",""...",scrub,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df[['lat', 'lon', 'natural', 'geometry']].to_parquet(rf"E:\Data\query-earth\embeddings\osm\2026\natural.parquet")

### POI

In [ ]:
import pandas as pd
import geopandas as gpd

df = gpd.read_parquet(rf"E:\Data\query-earth\osm\2026\pois.parquet")

In [ ]:
df = df[['lat', 'lon', 'amenity', 'name', 'geometry']]

In [ ]:
condition = df['name'].notna() | df['amenity'].notna()
result_df = df.loc[condition, ['lat', 'lon', 'amenity', 'name', 'geometry']]

In [ ]:
result_df

,lat,lon,amenity,name,geometry
0,34.948373,-116.864611,NaN,Calico Ghost Town,POINT (-116.86461 34.94837)
1,36.701463,-118.755997,NaN,California,POINT (-118.756 36.70146)
2,36.979368,-122.020726,bank,Chase,POINT (-122.02073 36.97937)
3,33.977685,-118.448647,NaN,Marina del Rey,POINT (-118.44865 33.97768)
4,33.995044,-118.466887,NaN,Venice,POINT (-118.46689 33.99504)
...,...,...,...,...,...
969157,NaN,NaN,NaN,North Las Vegas Gateway,"MULTIPOLYGON (((-115.13483 36.19594, -115.1344..."
969160,NaN,NaN,NaN,Hemet Golf Club,"POLYGON ((-117.05567 33.74986, -117.0566 33.75..."
969162,NaN,NaN,NaN,RDA Winchester,"MULTIPOLYGON (((-117.1075 33.60751, -117.10679..."
969163,NaN,NaN,NaN,Abelia Sports Park,"POLYGON ((-117.10123 33.61639, -117.10123 33.6..."


In [ ]:
result_df.to_parquet(rf"E:\Data\query-earth\embeddings\osm\2026\pois.parquet")

### Roads

In [ ]:
gpd.read_parquet(rf"E:\Data\query-earth\embeddings\osm\2014\natural.parquet")

,lat,lon,natural,geometry
0,41.738128,-121.529074,peak,POINT (-121.52907 41.73813)
1,41.728373,-121.548889,cave_entrance,POINT (-121.54889 41.72837)
2,41.731862,-121.523006,cave_entrance,POINT (-121.52301 41.73186)
3,41.733325,-121.526261,cave_entrance,POINT (-121.52626 41.73332)
4,41.731126,-121.510769,cave_entrance,POINT (-121.51077 41.73113)
...,...,...,...,...
161147,NaN,NaN,water,"POLYGON ((-120.49727 39.28691, -120.49725 39.2..."
161148,NaN,NaN,water,"POLYGON ((-120.50379 39.28144, -120.50379 39.2..."
161149,NaN,NaN,water,"POLYGON ((-121.78459 38.52861, -121.78467 38.5..."
161150,NaN,NaN,water,"POLYGON ((-121.67757 38.5431, -121.6777 38.543..."


In [ ]:
import pandas as pd
import geopandas as gpd

df = gpd.read_parquet(rf"E:\Data\query-earth\osm\2026\roads.parquet")

In [ ]:
df.highway.unique()

<ArrowStringArray>
[          'service',          'motorway',       'residential',
             'track',          'tertiary',           'primary',
      'primary_link',         'secondary',     'motorway_link',
      'unclassified',            'busway',             'trunk',
    'secondary_link',             'steps',           'footway',
          'cycleway',        'trunk_link',              'path',
     'living_street',        'pedestrian',     'tertiary_link',
         'bridleway',              'road',  'residential_link',
             'minor',      'turning_loop',          'corridor',
          'elevator',     'emergency_bay',          'bus_stop',
      'service;path',            'escape',           'disused',
      'bus_guideway',       'via_ferrata',          'crossing',
    'traffic_island',     'passing_place',          'scramble',
          'footpath',            'ladder',    'turning_circle',
 'footway:abandoned']
Length: 43, dtype: str

In [ ]:
df[["highway", "geometry"]].to_parquet(rf"E:\Data\query-earth\embeddings\osm\2026\roads.parquet")

### Waterways

In [ ]:
import pandas as pd
import geopandas as gpd

df = gpd.read_parquet(rf"E:\Data\query-earth\osm\2026\waterways.parquet")

In [ ]:
df

,lat,timestamp,version,visible,lon,changeset,id,tags,dock,water_point,waterfall,waterway,geometry,osm_type,dam,drain,stream
0,34.195230,1735753246,36,False,-118.601831,0.0,36395349,"{""description"":""The nominal (name only) source...",NaN,NaN,NaN,confluence,POINT (-118.60183 34.19523),node,NaN,NaN,NaN
1,34.139095,1351286856,4,False,-115.120558,0.0,54283836,NaN,NaN,NaN,NaN,weir,POINT (-115.12056 34.13909),node,NaN,NaN,NaN
2,33.896622,1769737191,6,False,-117.586105,0.0,54478232,"{""access"":""yes"",""barrier"":""gate"",""gate:type"":""...",NaN,NaN,NaN,floodgate,POINT (-117.58611 33.89662),node,NaN,NaN,NaN
3,36.145761,1766407508,2,False,-114.416950,0.0,158798380,NaN,NaN,NaN,NaN,confluence,POINT (-114.41695 36.14576),node,NaN,NaN,NaN
4,37.727553,1694886109,16,False,-119.543748,0.0,243671668,"{""alt_name"":""Yonapah"",""ele"":""1538"",""height"":""9...",NaN,NaN,NaN,waterfall,POINT (-119.54375 37.72755),node,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
972364,NaN,1779837918,1,None,NaN,0.0,20750416,"{""name"":""Caldwell Creek"",""type"":""waterway"",""me...",NaN,NaN,NaN,stream,"LINESTRING (-118.33841 35.78525, -118.33833 35...",relation,NaN,NaN,NaN
972365,NaN,1784206454,7,None,NaN,0.0,21106986,"{""name"":""Bear River"",""type"":""waterway"",""wikida...",NaN,NaN,NaN,river,"MULTILINESTRING ((-124.07594 40.38425, -124.07...",relation,NaN,NaN,NaN
972366,NaN,1784237544,1,None,NaN,0.0,21108733,"{""name"":""Big River"",""type"":""waterway"",""wikidat...",NaN,NaN,NaN,river,"LINESTRING (-123.38128 39.31517, -123.38183 39...",relation,NaN,NaN,NaN
972367,NaN,1784416506,3,None,NaN,0.0,21116252,"{""name"":""Gage Canal"",""type"":""waterway"",""wikida...",NaN,NaN,NaN,canal,"MULTILINESTRING ((-117.33622 33.97197, -117.33...",relation,NaN,NaN,NaN


In [ ]:
pd.read_parquet(rf"E:\Data\query-earth\embeddings\osm\2026\waterway.parquet")

,waterway,geometry
0,confluence,b'\x01\x01\x00\x00\x00\xda.~d\x84\xa6]\xc0s\xd...
1,weir,b'\x01\x01\x00\x00\x00[\xd1\xe68\xb7\xc7\\\xc0...
2,floodgate,b'\x01\x01\x00\x00\x004!\xf7\xbe\x82e]\xc0\xe3...
3,confluence,b'\x01\x01\x00\x00\x00M\x84\rO\xaf\x9a\\\xc0!\...
4,waterfall,b'\x01\x01\x00\x00\x00\xd3\xd1\xbb\xc2\xcc\xe2...
...,...,...
972364,stream,b'\x01\x02\x00\x00\x00\xa0\x01\x00\x00n+b}\xa8...
972365,river,b'\x01\x05\x00\x00\x00\x02\x00\x00\x00\x01\x02...
972366,river,b'\x01\x02\x00\x00\x00\x16\x04\x00\x00\x96\x8b...
972367,canal,b'\x01\x05\x00\x00\x00\x02\x00\x00\x00\x01\x02...


In [ ]:
df[['waterway', 'geometry']].to_parquet(rf"E:\Data\query-earth\embeddings\osm\2026\waterway.parquet")

---

## Appdx

In [3]:
import pandas as pd

df = pd.read_parquet(rf"E:\Data\query-earth\embeddings\osm\2026\buildings.parquet")

In [4]:
df

,amenity,name,geometry
0,NaN,Hangar 1,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x1f\x00...
1,NaN,NaN,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x14\x00..."
2,NaN,NaN,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
3,NaN,NaN,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\n\x00\x...
4,NaN,NaN,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\r\x00\x..."
...,...,...,...
10277113,NaN,NaN,b'\x01\x06\x00\x00\x00\x05\x00\x00\x00\x01\x03...
10277114,NaN,Augie's,"b""\x01\x03\x00\x00\x00\x02\x00\x00\x00\x05\x00..."
10277115,NaN,NaN,b'\x01\x03\x00\x00\x00\x02\x00\x00\x00&\x00\x0...
10277116,NaN,NaN,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x0e\x00...


In [5]:
df.amenity.unique()

<ArrowStringArray>
[                         nan,                    'parking',
                    'library',                    'toilets',
                   'pharmacy',                       'bank',
            'social_facility',                     'police',
                 'grave_yard',           'place_of_worship',
 ...
                     'gazebo',                     'cabana',
               'registration',                  'reception',
                 'egg-laying',               'event_center',
 'place_of_worship;monastery',               'tool_library',
    'school;place_of_worship',                  'cafe;fuel']
Length: 241, dtype: str